# Lesson 4: Diffusion samplers

A diffusion sampler replaces the single Gaussian from Lessons 1 and 2 by many simple Gaussian transitions. The forward variance-preserving process removes structure,

$$dX_t=-\tfrac12\beta(t)X_t\,dt+\sigma_K\sqrt{\beta(t)}\,dW_t,$$

and a learned reverse process maps the broad Gaussian prior $\mathcal N(0,30^2I)$ back to the GMM-40 target. The scale $\sigma_K=30$ matches the benchmark geometry. Your task is to implement the two SDE steps and the log probability of each transition kernel. The reparameterization and log-derivative estimators are already implemented.

## Setup and experiment flags

The default is a fixed 24-step linear VP schedule. Set <code>LEARN_DIFFUSION_SCHEDULE=True</code> to learn its monotone interior points while retaining suitable fixed endpoints. Training uses an entropy curriculum from $T=2$ to the intended $T=1$ target. We run the full $2\times2$ comparison: reparameterization versus log derivative, each with and without Langevin preconditioning. The preconditioned score correction follows Appendix D.4.3 of DDS: $f_\theta=\operatorname{clip}(\mathrm{NN}_1+\mathrm{NN}_2\odot\operatorname{clip}(\nabla\log p_T,-10^2,10^2),-10^4,10^4)$, with the target gradient detached.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.distributions import Independent, Normal

lesson_directory = Path("L4-DiffusionSamplers")
if not lesson_directory.is_dir():
    lesson_directory = Path(".")
sys.path.insert(0, str(lesson_directory.resolve()))

from lesson4_common import (
    BETA_END,
    BETA_START,
    DiffusionKernels,
    NUM_DIFFUSION_STEPS,
    PRIOR_STD,
    check_kernel_functions,
    evaluate_mode_recovery,
    plot_reward_landscape,
    plot_training_summary,
    save_training_comparison_animation,
    train_sampler,
)

torch.manual_seed(11)
LEARN_DIFFUSION_SCHEDULE = False
NUM_GIF_SAMPLES = 600
TRAINING_STEPS = 800
LOG_EVERY = 40
EXPERIMENTS = [
    (estimator, use_langevin)
    for estimator in ("reparameterization", "log_derivative")
    for use_langevin in (False, True)
]
print(f"VP steps: {NUM_DIFFUSION_STEPS}, beta: {BETA_START:.4f} -> {BETA_END:.4f}")
print(f"Prior standard deviation: {PRIOR_STD:g}; experiments: {EXPERIMENTS}")

## Target and prior

The target is the equal-weight GMM-40 benchmark: PyTorch seed 0, 40 means sampled uniformly from $[-40,40]^2$, and component covariance $I$. The terminal prior is $q_K=\mathcal N(0,30^2I)$.

In [ ]:
fig, axis = plt.subplots(figsize=(6.4, 5.2))
plot_reward_landscape(axis)
axis.legend(loc="lower right", fontsize=8)
axis.set_title("Lesson 4 target: seed-0 GMM-40")
plt.tight_layout()
plt.show()

## Task 1: implement the forward VP-SDE step

For $\alpha_k=1-\beta_k$,

$$x^k=\sqrt{\alpha_k}\,x^{k-1}+\sigma_K\sqrt{\beta_k}\,\varepsilon_k,\qquad \varepsilon_k\sim\mathcal N(0,I),\quad\sigma_K=30.$$

Use a diagonal Gaussian. Return the supplied fixed-noise transformation when <code>noise</code> is not <code>None</code>; otherwise choose <code>rsample</code> or <code>sample</code> using the flag.

In [ ]:
def forward_sde_step(
    previous_state: torch.Tensor,
    beta: torch.Tensor,
    *,
    reparameterize: bool,
    noise: torch.Tensor | None = None,
) -> torch.Tensor:
    """One variance-preserving forward noising step."""
    # TODO: Set alpha = 1 - beta; the std is PRIOR_STD * sqrt(beta).
    # TODO: If fixed noise is supplied, return mean + std * noise.
    # TODO: Otherwise sample from the diagonal Gaussian with rsample or sample.
    raise NotImplementedError("Implement the forward VP-SDE step")

## Task 2: implement the learned reverse VP-SDE step

Use the time-conditioned score in

$$q_\theta(x^{k-1}\mid x^k)=\mathcal N\!\left(\frac{x^k+\beta_k\sigma_K^2 s_\theta(x^k,k)}{\sqrt{1-\beta_k}},\,\beta_k\sigma_K^2 I\right).$$

In [ ]:
def reverse_sde_step(
    score_network,
    noisy_state: torch.Tensor,
    step_index: int,
    beta: torch.Tensor,
    num_steps: int,
    *,
    reparameterize: bool,
    noise: torch.Tensor | None = None,
) -> torch.Tensor:
    """One learned reverse denoising step."""
    # TODO: Evaluate the score network at noisy_state and this step.
    # TODO: Construct the reverse Gaussian mean and standard deviation.
    # TODO: Handle fixed noise, rsample, and sample as in the forward step.
    raise NotImplementedError("Implement the reverse VP-SDE step")

## Task 3: implement the forward-kernel log probability

Evaluate $\log p(x^k\mid x^{k-1})$ under the same Gaussian used by the forward step.

In [ ]:
def forward_kernel_log_prob(
    previous_state: torch.Tensor,
    noisy_state: torch.Tensor,
    beta: torch.Tensor,
) -> torch.Tensor:
    """Evaluate log p(x^k | x^{k-1})."""
    # TODO: Reconstruct the forward Gaussian and evaluate noisy_state.
    raise NotImplementedError("Implement the forward-kernel log probability")

## Task 4: implement the reverse-kernel log probability

Evaluate $\log q_\theta(x^{k-1}\mid x^k)$ using exactly the same score-conditioned mean as the reverse step.

In [ ]:
def reverse_kernel_log_prob(
    score_network,
    previous_state: torch.Tensor,
    noisy_state: torch.Tensor,
    step_index: int,
    beta: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    """Evaluate log q_theta(x^{k-1} | x^k)."""
    # TODO: Reconstruct the reverse Gaussian and evaluate previous_state.
    raise NotImplementedError("Implement the reverse-kernel log probability")

## Check the implementations

These checks verify shapes, finite values, and gradients through the score network and schedule.

In [ ]:
kernels = DiffusionKernels(
    forward_sde_step=forward_sde_step,
    reverse_sde_step=reverse_sde_step,
    forward_kernel_log_prob=forward_kernel_log_prob,
    reverse_kernel_log_prob=reverse_kernel_log_prob,
)
check_kernel_functions(kernels)

## Path-space objective: already implemented

Marginalizing a diffusion path is a data-processing operation, so

$$\mathrm{KL}(q_\theta(x^0)\|p_T(x^0))\leq\mathrm{KL}(q_\theta(x^{0:K})\|p_T(x^{0:K})).$$

The shared code expands this tractable joint KL into terminal negative reward plus forward/reverse log-kernel ratios. Both the pathwise and leave-one-out score estimators from Lesson 1 are already complete.

## Train the four sampler configurations

In [ ]:
comparison = []
trained_samplers = {}
for gradient_estimator, use_langevin in EXPERIMENTS:
    sampler, history = train_sampler(
        kernels,
        gradient_estimator=gradient_estimator,
        learn_schedule=LEARN_DIFFUSION_SCHEDULE,
        use_langevin_preconditioning=use_langevin,
        seed=11,
        steps=TRAINING_STEPS,
        log_every=LOG_EVERY,
        num_animation_samples=NUM_GIF_SAMPLES,
    )
    comparison.append((gradient_estimator, use_langevin, history))
    trained_samplers[(gradient_estimator, use_langevin)] = sampler
    _, recovered_modes = evaluate_mode_recovery(sampler, num_samples=5000)
    label = f"{gradient_estimator:18s} | Langevin={str(use_langevin):5s}"
    print(
        f"{label} | reward {history['mean_reward'][0]:7.2f} -> "
        f"{history['mean_reward'][-1]:7.2f} | recovered {recovered_modes.sum().item():2d}/40"
    )

reference_sampler = trained_samplers[("reparameterization", True)]
reference_history = next(
    history for estimator, langevin, history in comparison
    if estimator == "reparameterization" and langevin
)
plot_training_summary(reference_sampler, reference_history)
plt.show()

## Animate samples over training

Fixed prior and transition noise make each dot move smoothly between frames and make the four panels directly comparable. The GIF shows reparameterization and log derivative, each with and without Langevin preconditioning. The two loss panels use the same fixed-noise path objective for a fair comparison; their dashed vertical lines and point markers track the current frame.

In [ ]:
output_directory = Path("L4-DiffusionSamplers")
if not output_directory.is_dir():
    output_directory = Path(".")
gif_path = output_directory / "diffusion_sampler_training.gif"
save_training_comparison_animation(comparison, gif_path)
print(f"Saved animation to {gif_path.resolve()}")

from IPython.display import Image, display
display(Image(filename=str(gif_path)))